In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/train.csv")
y = train["PitNextLap"].astype(int).values

oof_001 = np.load("/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260501_001_lgb_strict_raw_baseline.npy")
oof_002 = np.load("/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260501_002_cat_strict_raw_baseline.npy")
oof_003 = np.load("/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260501_003_lgb_compound_tyrelife_min.npy")
oof_004 = np.load("/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260501_004_lgb_lap_race_time_norm_min.npy")
oof_006 = np.load("/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260501_006_cat_original_prior_min_without_normalized.npy")

def report(name, oof):
    print(name)
    print("  AUC:", roc_auc_score(y, oof))
    print()

report("001_lgb_raw", oof_001)
report("002_cat_raw", oof_002)
report("003_lgb_compound_tyrelife", oof_003)
report("004_lgb_lap_race_time_norm", oof_004)
report("006_cat_original_prior_min", oof_006)

pairs = [
    ("001", oof_001, "006", oof_006),
    ("002", oof_002, "006", oof_006),
    ("003", oof_003, "006", oof_006),
    ("004", oof_004, "006", oof_006),
]

for a_name, a, b_name, b in pairs:
    pearson = np.corrcoef(a, b)[0, 1]
    spearman = spearmanr(a, b).correlation
    print(f"{a_name} vs {b_name}: pearson={pearson:.6f}, spearman={spearman:.6f}")

blend_candidates = {
    "avg_002_006": (oof_002 + oof_006) / 2,
    "avg_001_002_006": (oof_001 + oof_002 + oof_006) / 3,
    "avg_002_003_004_006": (oof_002 + oof_003 + oof_004 + oof_006) / 4,
    "avg_001_002_003_004_006": (oof_001 + oof_002 + oof_003 + oof_004 + oof_006) / 5,
}

for name, o in blend_candidates.items():
    print(name, roc_auc_score(y, o))

001_lgb_raw
  AUC: 0.940723081019204

002_cat_raw
  AUC: 0.9485020378244504

003_lgb_compound_tyrelife
  AUC: 0.9413109097188121

004_lgb_lap_race_time_norm
  AUC: 0.9416811403305019

006_cat_original_prior_min
  AUC: 0.9487844854601896

001 vs 006: pearson=0.969194, spearman=0.961583
002 vs 006: pearson=0.995886, spearman=0.994109
003 vs 006: pearson=0.970549, spearman=0.961640
004 vs 006: pearson=0.970890, spearman=0.962230
avg_002_006 0.9489353487187131
avg_001_002_006 0.9480163006292464
avg_002_003_004_006 0.9473332843401526
avg_001_002_003_004_006 0.9465178047346736
